# Crime Incidents redux

We are going to again look at the crime data from the lesson five assignment. Ideally I would find data on the internet and have you import that, but I'm finding that open internet, permissionless resources for very large datasets are unreliable, so we're going to re-use the data from class 5 assignment and pretent that it is a very large dataset that we have to process lazily.

## Read the 'Crime_Incidents...' data into a single (lazy) dataframe then find the number of rows/records

Hint: For reading in you might want to use string substitution. The f-string `f"{some_number:02d}"` will zero pad a number so that the number 1, for instance, prints as '01'. Or alternatively you could use the `glob` package.

### Dask

In [99]:
import dask.dataframe as dd

In [100]:
from glob import glob
ddfs = []
files = glob("../Class_05/Assignment/Data/*.csv")

dtype={'CENSUS_TRACT': 'float64',
       'DISTRICT': 'float64',
       'PSA': 'float64',
       'XBLOCK': 'float64',
       'YBLOCK': 'float64',
       'WARD':'float64',
       'BID':'object',
      }

for i in files:
    tmp = dd.read_csv(i, dtype=dtype)
    ddfs.append(tmp)

In [101]:
ddf = dd.concat(ddfs)

In [102]:
ddf.columns

Index(['X', 'Y', 'CCN', 'REPORT_DAT', 'SHIFT', 'METHOD', 'OFFENSE', 'BLOCK',
       'XBLOCK', 'YBLOCK', 'WARD', 'ANC', 'DISTRICT', 'PSA',
       'NEIGHBORHOOD_CLUSTER', 'BLOCK_GROUP', 'CENSUS_TRACT',
       'VOTING_PRECINCT', 'LATITUDE', 'LONGITUDE', 'BID', 'START_DATE',
       'END_DATE', 'OBJECTID', 'OCTO_RECORD_ID'],
      dtype='object')

In [103]:
ddf['X'].count().compute()

np.int64(562317)

### Polars

In [107]:
import polars as pl

In [108]:
dft = pl.scan_csv(files[0])

In [109]:
dtypes = dft.head(3).collect().dtypes
names = dft.collect_schema().names()

In [110]:
dtp = dict(zip(names,dtypes))

In [111]:
dfs = []
for i in files:
    tmp = pl.scan_csv(i, schema=dtp)
    dfs.append(tmp)

In [112]:
df = pl.concat(dfs)

In [113]:
df.select('X').count().collect()

X
u32
562317


## Give return the rows of the three crimes closest to the capitol building.

The coordinates of the US capitol are latitude: 38.8900; longitude: -77.0091.

### DASK

In [104]:
ddf.head(3)

,X,Y,CCN,REPORT_DAT,SHIFT,METHOD,OFFENSE,BLOCK,XBLOCK,YBLOCK,...,BLOCK_GROUP,CENSUS_TRACT,VOTING_PRECINCT,LATITUDE,LONGITUDE,BID,START_DATE,END_DATE,OBJECTID,OCTO_RECORD_ID
0,-76.967206,38.872074,8175440,2008/12/12 22:00:00+00,EVENING,OTHERS,THEFT/OTHER,2800 - 2821 BLOCK OF PENNSYLVANIA AVENUE SE,402846.0,133805.0,...,007604 3,7604.0,Precinct 111,38.872067,-76.967204,<NA>,2008/12/12 17:00:00+00,2008/12/12 19:00:00+00,403455675,NaN
1,-76.998459,38.839721,8175469,2008/12/12 21:45:00+00,EVENING,OTHERS,ASSAULT W/DANGEROUS WEAPON,3301 - 3699 BLOCK OF 6TH STREET SE,400134.0,130213.0,...,009804 2,9804.0,Precinct 122,38.839713,-76.998457,<NA>,2008/12/12 20:03:00+00,2008/12/12 05:00:00+00,403455676,NaN
2,-77.037529,38.903747,8175472,2008/12/12 21:28:00+00,EVENING,OTHERS,THEFT/OTHER,1600 - 1699 BLOCK OF L STREET NW,396745.0,137321.0,...,010700 1,10700.0,Precinct 17,38.903739,-77.037526,GOLDEN TRIANGLE,2008/12/12 16:30:00+00,2008/12/12 20:30:00+00,403455677,NaN


In [105]:
ddf['distance'] = (ddf.X + 77.0091)**2 + (ddf.Y - 38.8900)**2

In [106]:
ddf.sort_values('distance').compute().head(3)

,X,Y,CCN,REPORT_DAT,SHIFT,METHOD,OFFENSE,BLOCK,XBLOCK,YBLOCK,...,CENSUS_TRACT,VOTING_PRECINCT,LATITUDE,LONGITUDE,BID,START_DATE,END_DATE,OBJECTID,OCTO_RECORD_ID,distance
8075,-77.008647,38.892059,23420250,2023/01/17 15:02:36+00,DAY,OTHERS,THEFT F/AUTO,1 - 2 BLOCK OF CONSTITUTION AVENUE NE,399250.0,136023.0,...,980000.0,Precinct 130,38.892052,-77.008645,CAPITOL HILL,2023/01/11 20:45:00+00,2023/01/12 17:25:00+00,476483318,NaN,0.000004
29844,-77.009558,38.892086,10180707,2010/12/16 12:15:00+00,DAY,OTHERS,THEFT/OTHER,1 - 3 BLOCK OF CONSTITUTION AVENUE NW,399171.0,136026.0,...,6202.0,Precinct 129,38.892079,-77.009556,CAPITOL HILL,2010/12/15 23:30:00+00,2010/12/16 12:15:00+00,403188864,NaN,0.000005
8447,-77.008624,38.887591,10087404,2010/06/22 20:10:00+00,EVENING,OTHERS,THEFT/OTHER,1 - 15 BLOCK OF INDEPENDENCE AVENUE SE,399252.0,135527.0,...,6202.0,Precinct 130,38.887583,-77.008622,CAPITOL HILL,2010/06/22 15:55:00+00,2010/06/22 19:45:00+00,402977466,NaN,0.000006


### Polars

In [115]:
df.with_columns(
    ((pl.col('X') + 77.0091)**2 + (pl.col('Y') - 38.8900)**2).alias('distance')
).sort(by='distance').head(3).collect()

X,Y,CCN,REPORT_DAT,SHIFT,METHOD,OFFENSE,BLOCK,XBLOCK,YBLOCK,WARD,ANC,DISTRICT,PSA,NEIGHBORHOOD_CLUSTER,BLOCK_GROUP,CENSUS_TRACT,VOTING_PRECINCT,LATITUDE,LONGITUDE,BID,START_DATE,END_DATE,OBJECTID,OCTO_RECORD_ID,distance
f64,f64,i64,str,str,str,str,str,f64,f64,i64,str,i64,i64,str,str,i64,str,f64,f64,str,str,str,i64,str,f64
-77.008647,38.892059,23420250,"""2023/01/17 15:02:36+00""","""DAY""","""OTHERS""","""THEFT F/AUTO""","""1 - 2 BLOCK OF CONSTITUTION AV…",399250.0,136023.0,6,"""6C""",1,102,"""Cluster 45""","""980000 1""",980000,"""Precinct 130""",38.892052,-77.008645,"""CAPITOL HILL""","""2023/01/11 20:45:00+00""","""2023/01/12 17:25:00+00""",476483318,null,0.000004
-77.009558,38.892086,10180707,"""2010/12/16 12:15:00+00""","""DAY""","""OTHERS""","""THEFT/OTHER""","""1 - 3 BLOCK OF CONSTITUTION AV…",399171.0,136026.0,2,"""2C""",1,104,null,"""006202 1""",6202,"""Precinct 129""",38.892079,-77.009556,"""CAPITOL HILL""","""2010/12/15 23:30:00+00""","""2010/12/16 12:15:00+00""",403188864,null,0.000005
-77.008624,38.887591,9049956,"""2009/04/15 14:49:00+00""","""DAY""","""OTHERS""","""ASSAULT W/DANGEROUS WEAPON""","""1 - 15 BLOCK OF INDEPENDENCE A…",399252.0,135527.0,6,"""6B""",1,106,"""Cluster 26""","""006202 1""",6202,"""Precinct 130""",38.887583,-77.008622,"""CAPITOL HILL""","""2009/04/15 14:49:00+00""","""2009/04/15 04:00:00+00""",408134220,null,0.000006
